# 2일차 실습 노트북: 실습 미션 1, 2, 3
**실습 미션 세 개가 들어 있습니다. 미션마다 기본 실행 → 바꿔 보기 A, B → Q1, Q2 답 적기 순서로 진행합니다.**


이 노트북에는 실습 미션 세 개가 들어 있습니다. 코드 셀은 모두 완성되어 있어서 그대로 실행하면 됩니다.
세 미션 모두 아래 순서로 진행합니다.
1. **기본 실행**: 미션의 (1)번 셀부터 차례로 실행합니다.
2. **바꿔 보기 A, B**: 바꿔 보기 셀에는 바꾼 값이 이미 들어 있습니다. 셀을 그대로 실행하고, 기본 실행 때와 결과가 어떻게 다른지 봅니다. 셀 맨 위 한 줄의 값을 다른 값으로 바꿔 더 해 볼 수도 있습니다.
3. **Q1, Q2 답 적기**: 질문 바로 아래에 있는 답 셀에서 「(여기에 적습니다)」를 지우고 답을 적습니다. 답에는 출력에 나온 숫자를 근거로 넣습니다. 답 셀은 두 번 클릭하면 고칠 수 있고, 질문 셀은 고치지 않습니다.
4. **확장 미션(심화)**: 목적만 받고 어떻게 만들지 스스로 정해, PC 의 Claude Code 로 직접 만들어 봅니다. 만들어 본 소감은 미션 끝의 「회고」 아래 답 셀에 적습니다.

**✏️ 표시 보는 법** · 실습 미션 도중 고쳐야 하는 칸에는 ✏️ 가 붙어 있습니다.
- **✏️ 직접 고칠 곳**: 꼭 고쳐야 하는 칸입니다(미션 3 의 문서 이름 · 판정).
- **✏️ 더 해 보기(선택)**: 지금 들어 있는 값으로 먼저 실행합니다. 기본 미션을 마친 뒤 값을 다르게 바꿔 다시 실행하면 결과가 어떻게 달라지는지 볼 수 있습니다.
- **✏️ 답 적는 곳**: Q1, Q2 와 확장 미션 회고 아래의 답 셀입니다.

코드 안에서는 `# ✏️ ========` 두 줄 사이가 손대는 칸입니다.

예제에서 만든 함수는 각 미션 앞의 「준비」 셀이 다시 만들어 줍니다. 그래서 예제 노트북을 함께 열어 두지 않아도 됩니다.

**처음 한 번만 할 일**은 예제 노트북과 같습니다. [파일] → [Drive에 사본 저장] 을 눌러 내 드라이브에 사본을 만듭니다. API 키는 예제 노트북에서 저장한 보안 비밀을 그대로 쓰므로, 이 노트북에서는 왼쪽 열쇠 아이콘을 눌러 「노트북 액세스」만 다시 켭니다.
그다음 「환경 준비」 셀 세 개를 실행하고, 하려는 미션의 「준비」 셀부터 차례로 실행합니다.

## 환경 준비

**실습 재료를 내려받고 패키지를 설치합니다.**

이 셀은 학습 사이트에서 실습 재료 ZIP 파일을 내려받아 지금 폴더에 풀고, 코랩에 없는 패키지를 설치합니다. 파일을 직접 올리거나 압축을 풀 필요는 없습니다.
- `anthropic` 은 파이썬에서 Claude API 를 호출할 때 쓰는 공식 SDK(개발 도구 묶음)입니다. `openpyxl` 은 pandas 가 엑셀 파일을 저장할 때 쓰는 패키지이고, `sentence-transformers` 는 미션 3 에서 문장을 벡터(임베딩)로 바꾸는 라이브러리입니다.
- `%pip` 는 노트북 안에서 pip 설치를 실행하는 명령이고, `-q` 를 붙이면 설치 과정의 출력이 줄어듭니다.
- 폴더에 `inquiries_120.csv` 가 이미 있으면 재료를 다시 내려받지 않습니다.

출력된 CSV 파일 목록에 `inquiries_120.csv` 가 보이면 다음 셀로 넘어갑니다.

In [ ]:
# 환경 준비 (1) 실습 재료를 내려받고 패키지를 설치합니다 · 처음 한 번만
%pip install -q anthropic openpyxl sentence-transformers
import os, urllib.request, zipfile
URL = 'https://lecture-web-five.vercel.app/materials/day02-data.zip'
if not os.path.exists('inquiries_120.csv'):
    urllib.request.urlretrieve(URL, 'day02-data.zip')
    zipfile.ZipFile('day02-data.zip').extractall('.')
print(sorted(f for f in os.listdir('.') if f.endswith('.csv')))

**코랩 보안 비밀에 저장한 API 키를 불러옵니다.**

이 셀은 코랩 보안 비밀에 저장해 둔 API 키를 읽어서 환경 변수 `ANTHROPIC_API_KEY` 에 넣습니다. 보안 비밀에 저장한 값은 환경 변수로 자동으로 옮겨지지 않습니다. 그래서 이 셀을 건너뛰면 다음 셀에서 키가 없다는 오류가 납니다. 키를 코드에 직접 적어 두면 노트북을 공유할 때 키도 함께 공유되므로, 코드에는 적지 않습니다.
- **환경 변수**는 프로그램이 실행되는 동안 읽을 수 있는 이름과 값의 쌍입니다. SDK 는 `ANTHROPIC_API_KEY` 라는 이름에 들어 있는 값을 API 키로 씁니다.
- **try / except** 는 `try` 아래 코드에서 오류(예외)가 나도 셀을 멈추지 않고 `except` 아래 코드를 실행하게 하는 문법입니다. 예외 처리는 예제 1-3 에서 자세히 다룹니다.
- 처음 실행하면 「데이터 접근」을 허용할지 묻는 창이 뜹니다. [허용] 을 누릅니다.

출력에 나온 키 앞부분이 `sk-ant-` 로 시작하는지 확인합니다. 키를 읽지 못하면 무엇을 확인해야 하는지 세 줄로 출력됩니다.

In [ ]:
# 환경 준비 (2) 보안 비밀의 키를 환경 변수로 옮깁니다 · 키를 코드에 적지 않습니다
import os
from google.colab import userdata                 # 코랩 보안 비밀

try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY').strip()
    print('키 앞부분', os.environ['ANTHROPIC_API_KEY'][:7], '· 길이', len(os.environ['ANTHROPIC_API_KEY']))
except Exception as e:
    print('보안 비밀에서 키를 읽지 못했습니다:', type(e).__name__)
    print('1. 열쇠 아이콘의 이름이 ANTHROPIC_API_KEY 와 같은지 봅니다')
    print('2. 그 줄의 「노트북 액세스」가 켜져 있는지 봅니다')
    print('3. 「데이터 접근」 창이 떴다면 [허용]을 누릅니다')
    print('확인한 뒤 이 셀을 다시 실행합니다')

**클라이언트를 만들고 연결을 확인합니다.**

이 셀은 API 를 호출할 클라이언트를 만들고, 오늘 쓸 모델 이름을 변수에 담습니다. 그다음 경량 모델에 짧은 요청을 한 번 보내 연결이 되는지 확인합니다.
- **client** 는 API 서버와 주고받는 일을 맡는 객체입니다. 앞으로 모델 호출은 모두 `client.messages.create(...)` 형태로 합니다.
- **모델 등급**: 경량 모델(haiku)은 빠르고 요금이 낮습니다. 대형 모델(opus)은 느리고 요금이 높습니다. 중형 모델(sonnet)은 그 중간입니다.
- **messages.create** 는 모델에 요청을 보내는 함수입니다. `model`, `max_tokens`, `messages` 를 넣으며, 각 값의 뜻은 예제 1-1 에서 하나씩 살펴봅니다.
- **text_of()** 는 응답에서 글자만 꺼내는 함수입니다. 응답 본문은 여러 블록의 목록이고 글자 블록(`text`) 말고 다른 종류의 블록이 섞일 수 있어서, `text` 블록만 골라 이어 붙입니다.

SDK 버전과 `준비 완료` 가 출력되면 환경 준비가 끝난 것입니다.

In [ ]:
# 환경 준비 (3) 클라이언트와 모델 이름을 정하고 연결을 확인합니다
import anthropic, json, time
import pandas as pd

client = anthropic.Anthropic()        # 환경 변수 ANTHROPIC_API_KEY 를 읽습니다
MODEL       = 'claude-sonnet-5-5'     # 중형 · 오늘의 기본 모델
SMALL_MODEL = 'claude-haiku-4-5'      # 경량 · 대량 분류용
BIG_MODEL   = 'claude-opus-5-5'       # 대형 · 비교용
print('anthropic', anthropic.__version__)

def text_of(r):
    # 응답의 content 는 블록 목록입니다. text 블록만 이어 붙입니다
    return ''.join(b.text for b in r.content if b.type == 'text')

r = client.messages.create(model=SMALL_MODEL, max_tokens=20,
        messages=[{'role': 'user', 'content': '"준비 완료" 라고만 답해줘.'}])
print(text_of(r))

## 미션 1, 2 준비: 예제에서 만든 함수를 다시 만듭니다
**예제에서 만든 함수를 다시 만드는 셀입니다. 실행만 합니다.**

예제 1-1 ~ 1-3 과 예제 2-2 에서 만든 분류 규칙, 요금표, 함수를 이 셀 하나에 모았습니다. 코드는 예제와 같고, 이 셀은 API 를 호출하지 않습니다.
- `SYS_CLS` 는 예제 1-1 (4), `count_in()` 은 예제 1-2 (2), `PRICE` 와 `OUT_TOK` 은 예제 1-2 (3), `run()` 은 예제 1-2 (4) 에서 만들었습니다.
- `ask()` 는 예제 1-3 (9), `check_response()` 는 예제 1-3 (11), `is_valid()` 는 예제 1-3 (12), `RUN_AT` 은 예제 2-2 (4) 에서 만들었습니다.

`준비 완료` 가 출력되면 미션 1 이나 미션 2 를 시작합니다.

In [ ]:
# 미션 1 · 2 준비 · 예제 1-1 ~ 2-2 의 규칙과 함수
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
import sqlite3

SYS_CLS = ('너는 반도체 공장 현장 문의를 분류하는 도우미다. '
           '[문서안내/절차안내/긴급/데이터요청] 네 가지 중 하나로만 분류한다. '
           '출력은 {"id":..., "category":..., "urgent":true/false} JSON 한 줄만.')      # 예제 1-1 (4)

def count_in(s):                                     # 예제 1-2 (2) · 빈 문의는 0
    if not s.strip():
        return 0
    return client.messages.count_tokens(model=SMALL_MODEL, system=SYS_CLS,
        messages=[{'role': 'user', 'content': s}]).input_tokens

PRICE = {SMALL_MODEL: (1, 5), MODEL: (2, 10), BIG_MODEL: (4, 20)}   # 예제 1-2 (3) · 100만 토큰당 달러 (입력, 출력)
OUT_TOK = 30

def run(model, x):                                   # 예제 1-2 (4) · 한 건 보내 (분류, 입력, 출력, 초)
    t0 = time.time()
    r = client.messages.create(model=model, max_tokens=1024, system=SYS_CLS,
            messages=[{'role': 'user', 'content': f"id: {x['id']}\n문의: {x['문의내용']}"}])
    t = text_of(r)
    cat = json.loads(t[t.find('{'):t.rfind('}') + 1])['category']
    return cat, r.usage.input_tokens, r.usage.output_tokens, time.time() - t0

# 예제 1-3 (9) · 다시 보낼 오류
RETRY = (anthropic.RateLimitError, anthropic.OverloadedError,
         anthropic.InternalServerError, anthropic.APIConnectionError)

def ask(system, user, model=SMALL_MODEL, max_tokens=300, tries=3, timeout=30, **extra):
    c = client.with_options(max_retries=0, timeout=timeout)
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role': 'user', 'content': user}], **extra)
    if system:
        kw['system'] = system
    for i in range(tries):
        try:
            return c.messages.create(**kw)
        except RETRY as e:
            if i == tries - 1:
                raise
            print(f'  {type(e).__name__} → {2 ** i}초 뒤 다시')
            time.sleep(2 ** i)

def check_response(r):                               # 예제 1-3 (11) · 예외 없는 실패
    if r.stop_reason == 'max_tokens':
        return '잘림'
    if r.stop_reason == 'refusal':
        return '거절'
    if '{' not in text_of(r):
        return 'JSON 없음'
    return None

def is_valid(s):                                     # 예제 1-3 (12) · 다섯 글자 미만은 거릅니다
    return isinstance(s, str) and len(s.strip()) >= 5

RUN_AT = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')   # 예제 2-2 (4) · 실행 시각
print('준비 완료')

## 실습 미션 1 · 요금을 세고 거른 뒤 끝까지 보내기
**문의 60건을 보내기 전에 요금을 확인하고, 보내면 안 되는 문의를 거른 뒤, 모두 보내 실제 요금과 비교합니다.**

재료는 `2일차_미션1_재료.csv` 이고 문의 60건이 들어 있습니다. 열은 `id` 와 `문의내용` 두 개입니다. 이 미션에서는 문의를 보내기 전에 토큰 수와 예상 요금을 세고, 보내면 안 되는 문의를 거릅니다. 그다음 남은 문의를 모두 보내고, 실제로 나온 요금을 예상 요금과 비교합니다.
예제 1-2 에서 만든 `count_in()` 과 `PRICE`, 예제 1-3 에서 만든 `is_valid()`, `ask()`, `check_response()` 를 그대로 씁니다.

**✏️ 이 미션에서 손댈 곳**
- 답 적는 곳: Q1, Q2
- 더 해 보기(선택): 바꿔 보기 A 의 `MAX_IN_A`, 바꿔 보기 B 의 `MAX_TOK_B`
- 실습 미션 1 (1)~(4) 셀(기본 실행)은 실행만 합니다.

**진행 순서**
1. **기본 실행**: (1)~(4) 셀을 차례로 실행합니다. 필요한 값은 모두 들어 있습니다.
2. **바꿔 보기 A, B**: 바꿔 보기 셀에는 바꾼 값이 이미 들어 있습니다. 셀을 그대로 실행하고, 기본 실행 때와 결과가 어떻게 다른지 봅니다.
3. **Q1, Q2 답 적기**: 질문 아래 답 셀의 「(여기에 적습니다)」를 지우고, 출력에 나온 숫자를 근거로 답을 적습니다.
4. **확장 미션(심화)**: 목적만 받고 문의 자동 분류 프로그램을 Claude Code 로 직접 만들어 봅니다.

**미션 재료 60건을 읽어 문의마다 입력 토큰 수를 세고, 모델 등급별로 예상 요금을 계산합니다.**

- **참고할 예제**: 예제 1-2 (2), (3)
- 아직 문의를 보내지 않았으므로 출력 토큰 수는 알 수 없습니다. 그래서 출력 토큰은 예제 1-2 (3) 에서 정한 가정값 `OUT_TOK` 으로 계산합니다.

In [ ]:
# 실습 미션 1 (1) 재료를 읽고 보내기 전에 토큰과 요금을 셉니다 · 예제 1-2 (2)(3)
m1 = pd.read_csv('2일차_미션1_재료.csv').fillna('')
with ThreadPoolExecutor(max_workers=4) as ex:
    m1['in_tok'] = list(ex.map(count_in, m1['문의내용']))   # 빈 문의는 0
print('건수', len(m1), '· 입력 합계', m1['in_tok'].sum(), '· 최대', m1['in_tok'].max())
calls = (m1['in_tok'] > 0).sum()          # 빈 문의를 뺀 호출 수
for m, (p_in, p_out) in PRICE.items():
    usd = (m1['in_tok'].sum() * p_in + calls * OUT_TOK * p_out) / 1_000_000
    print(f'{m:18s} 예상 ${usd:.4f}')

**보내기 전에 세 가지 문의를 차례로 거릅니다. 내용이 비어 있는 문의, 입력 토큰이 상한을 넘는 긴 문의, 연락처(메일 주소나 휴대폰 번호)가 들어 있는 문의입니다. 조건마다 몇 건이 걸러졌는지 따로 셉니다.**

- **참고할 예제**: 예제 1-3 (12) 에서 `is_valid()` 로 문의를 거른 코드
- 입력 상한 `MAX_IN` 은 300 토큰입니다.

In [ ]:
# 실습 미션 1 (2) 보내기 전에 거릅니다 · 조건마다 몇 건을 걸렀는지 따로 셉니다
MAX_IN = 300            # 입력 상한(토큰) · 바꿔 보기 A 에서 올려 봅니다

def no_contact(s):      # 메일 주소(@)도 휴대폰 번호(010-)도 없으면 True
    return '@' not in s and '010-' not in s

n0 = len(m1)
work1 = m1[m1['문의내용'].map(is_valid)];         n1 = len(work1)   # 빈 문의
work1 = work1[work1['in_tok'] <= MAX_IN];         n2 = len(work1)   # 상한 초과
work1 = work1[work1['문의내용'].map(no_contact)];  n3 = len(work1)   # 연락처
print('빈 문의', n0 - n1, '· 입력 상한 초과', n1 - n2, '· 연락처 포함', n2 - n3, '· 보낼 것', n3)

**걸러지고 남은 문의를 `ask()` 로 모두 보내고, 성공한 문의 목록, 실패한 문의 목록, 호출 기록(로그)을 모읍니다.**

- **참고할 예제**: 예제 1-3 (12) 의 반복문입니다. 이 반복문을 함수 `send_all()` 로 묶어 두었습니다.
- `ask()` 에 넘기는 값은 세 가지입니다. 응답 길이 상한은 `max_tokens=200`, 최대 시도 횟수는 `tries=3`, 한 번 호출할 때 기다리는 시간 제한은 `timeout=30`(초)입니다.
- 실패 목록에는 (id, 실패 사유) 짝이 들어갑니다. 실패 사유는 `check_response()` 가 돌려준 말(잘림, 거절, JSON 없음) 가운데 하나이거나, 호출 중에 난 오류의 이름입니다.

In [ ]:
# 실습 미션 1 (3) 남은 건을 끝까지 보냅니다 · 예제 1-3 (12) 와 같은 코드를 함수로 묶었습니다
ANSWER = {}                                  # 모델 답을 id 별로 모아 둡니다

def send_all(rows, max_tokens=200, tries=3, timeout=30):
    ok, failed, log = [], [], []
    for _, x in rows.iterrows():
        try:
            r = ask(SYS_CLS, f"id: {x['id']}\n문의: {x['문의내용']}",
                    max_tokens=max_tokens, tries=tries, timeout=timeout)
            ANSWER[x['id']] = text_of(r)
            why = check_response(r)          # 잘림 · 거절 · JSON 없음이면 사유
            if why:
                failed.append((x['id'], why))
            else:
                ok.append(x['id'])
            u = r.usage
            log.append((x['id'], r.model, u.input_tokens, u.output_tokens, r.stop_reason))
        except anthropic.APIError as e:      # 재시도 끝에 실패한 호출
            failed.append((x['id'], type(e).__name__))
    return ok, failed, log

ok, failed, log = send_all(work1)
print('보냄', len(work1), '· 성공', len(ok), '· 실패', len(failed), failed)

**실습 미션 1 (3) 에서 모은 로그의 입력 토큰과 출력 토큰을 더해 실제 요금을 계산하고, (1) 에서 계산한 예상 요금과 비교합니다.**

- **참고할 예제**: 예제 1-2 (5)
- 로그 한 줄에는 id, 모델, 입력 토큰, 출력 토큰, 종료 이유가 이 순서로 들어 있습니다. 종료 이유는 모델이 응답을 멈춘 까닭입니다. 답을 다 쓰고 끝냈으면 `end_turn`, 응답 길이 상한에 걸려 잘렸으면 `max_tokens` 입니다. 요금 계산은 함수 `cost()` 로 묶어 두었습니다.

In [ ]:
# 실습 미션 1 (4) 로그로 실제 요금을 계산해 (1) 의 예상과 비교합니다 · 예제 1-2 (5)
def cost(log):                                # 로그 다섯 칸 중 입력 · 출력 토큰
    tin, tout = sum(x[2] for x in log), sum(x[3] for x in log)
    return tin, tout, (tin * PRICE[SMALL_MODEL][0] + tout * PRICE[SMALL_MODEL][1]) / 1e6

tin, tout, usd = cost(log)
print(f'호출 {len(log)}건 · 입력 {tin} · 출력 {tout} · 실제 ${usd:.4f}')

### 바꿔 보기 A: 입력 상한을 올려 봅니다
**입력 상한을 700 으로 올린 셀을 그대로 실행하고, 거른 건수와 요금이 어떻게 바뀌는지 봅니다.**

입력 상한 `MAX_IN` 을 300 에서 700 으로 올리면, 기본 실행에서 걸러졌던 긴 문의가 어떻게 되는지 봅니다. 이 셀은 상한을 올려서 새로 통과한 문의만 보내고, 그 요금을 기본 실행 요금에 더해서 보여 줍니다.
**✏️ 더 해 보기(선택)**: 먼저 셀을 그대로 실행한 뒤, 셀 맨 위 한 줄에 있는 `MAX_IN_A` 의 값을 다른 숫자(예: 500, 1000)로 바꿔 다시 실행해 볼 수도 있습니다.

In [ ]:
# 실습 미션 1 바꿔 보기 A · 입력 상한을 바꿔 다시 거르고, 보내고, 요금을 셉니다
# ✏️ ======== 더 해 보기(선택) · 처음에는 그대로 실행, 그다음 값을 바꿔 다시 실행 ========
MAX_IN_A = 700          # 이 값을 바꿔 다시 실행합니다 · 기본 실행은 300
# ✏️ ======== 여기까지 ========
ok_rows = m1['문의내용'].map(is_valid) & m1['문의내용'].map(no_contact)
workA = m1[ok_rows & (m1['in_tok'] <= MAX_IN_A)]
print('입력 상한 초과', (ok_rows & (m1['in_tok'] > MAX_IN_A)).sum(), '· 보낼 것', len(workA), f'(기본 실행 {len(work1)})')
new = workA[~workA['id'].isin(work1['id'])]   # 상한을 올려 새로 보낼 문의
okA, failedA, logA = send_all(new)
tinA, toutA, usdA = cost(logA)
print(f'새로 보낸 {len(new)}건 · 입력 {tinA} · 출력 {toutA} · 추가 ${usdA:.4f}')
print(f'합계 ${usd + usdA:.4f} (기본 실행 ${usd:.4f}) · 실패 {failedA}')
m1i = m1.set_index('id')
for i in new['id']:                           # 긴 문의의 원문 앞부분과 받은 답
    print(i, m1i.loc[i, 'in_tok'], '토큰 · 원문 앞부분:', m1i.loc[i, '문의내용'][:150], '…')
    print('   → 받은 답', ANSWER.get(i, '(답 없음)').replace('\n', ' '))

**Q1.** 입력 상한 `MAX_IN` 을 300 에서 700 으로 올렸을 때의 결과를 보고 답해 주세요.
1. 거른 건수, 보낸 건수, 합계 요금은 각각 얼마에서 얼마로 바뀌었나요?
2. 새로 보낸 긴 문의 두 건(Q222, Q251)의 원문 앞부분과 모델이 돌려준 답을 읽어 보세요. 실제 업무라면 이런 긴 문의를 그대로 모델에 보낼 건가요, 아니면 다르게 처리할 건가요? 그렇게 판단한 이유도 함께 적어 주세요.

**답** · ✏️ 답 적는 곳 (두 번 클릭해 적기)

(여기에 적습니다)

### 바꿔 보기 B: 응답 길이 상한을 줄여 봅니다
**응답 길이 상한을 20 으로 줄인 셀을 그대로 실행하고, 성공과 실패가 어떻게 바뀌는지 봅니다.**

응답 길이 상한 `max_tokens` 를 200 에서 20 으로 줄이고, 실습 미션 1 (2) 에서 걸러지고 남은 문의 가운데 앞의 10건을 다시 보냅니다. 오류가 나는지, 그리고 성공 목록과 실패 목록이 어떻게 바뀌는지 봅니다.
**✏️ 더 해 보기(선택)**: 먼저 셀을 그대로 실행한 뒤, 셀 맨 위 한 줄에 있는 `MAX_TOK_B` 의 값을 다른 숫자(예: 40, 60)로 바꿔 다시 실행해 볼 수도 있습니다.

In [ ]:
# 실습 미션 1 바꿔 보기 B · 응답 길이 상한을 바꿔 앞 10건을 다시 보냅니다
# ✏️ ======== 더 해 보기(선택) · 처음에는 그대로 실행, 그다음 값을 바꿔 다시 실행 ========
MAX_TOK_B = 20          # 이 값을 바꿔 다시 실행합니다 · 기본 실행은 200
# ✏️ ======== 여기까지 ========
okB, failedB, logB = send_all(work1.head(10), max_tokens=MAX_TOK_B)
print('성공', len(okB), '· 실패', len(failedB), failedB)
for row in logB[:3]:                         # 로그 세 줄과 받은 답 앞부분
    print(' 로그', row, '·', repr(ANSWER[row[0]][:40]))
outs = [x[3] for x in log]                   # 기본 실행(max_tokens 200)의 한 건 출력 토큰
print('기본 실행 한 건 출력 토큰 · 최소', min(outs), '· 평균', round(sum(outs) / len(outs), 1), '· 최대', max(outs))

**Q2.** 응답 길이 상한 `max_tokens` 를 20 으로 줄였을 때의 결과를 보고 답해 주세요.
1. 오류(예외)가 났나요? 앞 10건 가운데 몇 건이 성공하고 몇 건이 실패했으며, 실패 사유는 무엇이었나요?
2. 그 실패를 알아낸 것은 어느 함수였나요?
3. 출력의 마지막 줄에 기본 실행 때 한 건당 출력 토큰 수(최소, 평균, 최대)가 나옵니다. 이 숫자를 근거로, 여러 건을 한꺼번에 보내는 작업에서 `max_tokens` 를 얼마로 두겠는지 적어 주세요.

**답** · ✏️ 답 적는 곳 (두 번 클릭해 적기)

(여기에 적습니다)

### 확장 미션 1 (심화) · 문의 자동 분류 프로그램 만들기
**기본 미션을 마친 경우에만 합니다. 목적만 보고 어떻게 만들지 스스로 정한 뒤, Claude Code 로 직접 만들어 봅니다.**

**목적**
매일 현장 문의가 수십 건씩 들어옵니다. 이 문의를 Claude 로 자동 분류해 담당자에게 넘길 수 있는 프로그램이 필요합니다. 보내기 전에 요금을 확인할 수 있어야 하고, 개인정보가 든 문의는 밖으로 보내지 않아야 하며, 중간에 몇 건이 실패해도 나머지는 처리되어야 합니다.

**재료**: `2일차_미션1_재료.csv` (코랩 왼쪽 파일 패널에서 ⋮ → [다운로드]) · 본인 업무에 맞게 재료를 새로 만들어 진행해도 됩니다.

**만들기 전에 생각해 볼 것**
- 보내기 전에 무엇을 세고, 무엇을 걸러야 할까요?
- 실패에는 어떤 종류가 있고, 각각 어떻게 다뤄야 할까요?
- 오늘 미션 1 에서 쓴 블록 가운데 무엇을 가져다 쓸 수 있을까요?

**진행**
1. 목적을 읽고, 어떤 단계로 만들지 먼저 짧게 메모합니다.
2. PC 에 작업 폴더를 만들고 재료 파일을 넣은 뒤, Claude Code 에서 그 폴더를 열어 **새 세션**을 시작합니다.
3. 목적과 내 계획을 Claude Code 에 보내 만들게 하고, 결과를 보며 고쳐 달라고 요청하면서 완성해 갑니다.
4. 다 만들면 맨 아래 「회고」에 직접 만들어 보니 어땠는지 적습니다.

**✏️ 회고** · 직접 만들어 보니 어땠는지 자유롭게 적습니다. 예를 들면 이런 것들입니다.
- 처음 세운 계획과 실제로 만들어진 구조가 어떻게 달랐는지
- Claude Code 에 다시 요청한 것과 그 까닭
- 오늘 미션의 코드와 다르게 만든 부분, 또는 미션 코드에서 가져다 쓴 부분

**답** · ✏️ 답 적는 곳 (두 번 클릭해 적기)

(여기에 적습니다)

## 실습 미션 2 · 스키마를 먼저 정하고 80건 일괄 처리
**접수 80건을 담당팀으로 나눠 엑셀과 DB 에 저장합니다.**

재료는 `2일차_미션2_재료.csv` 이고 접수 본문 80건이 들어 있습니다. 열은 `id`, `접수시각`, `본문` 세 개이고, 본문에는 설비, 시설, IT, 안전 요청이 섞여 있습니다. 이 미션에서는 모델에게서 받을 항목(스키마)을 먼저 정하고, 3건으로 결과를 확인한 뒤 80건 전체로 늘립니다.
예제 2-1 (3) 의 스키마, 예제 2-2 (2)~(6) 의 분류 함수, 엑셀 파일과 DB 에 저장하는 코드, 예제 2-3 (1), (2) 의 검증 코드를 조금 바꿔 씁니다.

**✏️ 이 미션에서 손댈 곳**
- 답 적는 곳: Q1, Q2
- 더 해 보기(선택): 실습 미션 2 (1) 셀의 `TEAMS`(담당팀 목록), `URGENT_RULE` · `SYS2`(분류 기준), `HAND`(직접 붙인 정답 라벨 10건), (4) 셀의 업무 규칙, 바꿔 보기 A 의 `DROP_TEAM`, 바꿔 보기 B 의 `DROP_RULE`

**진행 순서**
1. **기본 실행**: (1)~(5) 셀을 차례로 실행합니다. 코드와 값은 모두 들어 있습니다. (2) 에서 3건의 결과를 먼저 보고, 이상이 없으면 (3) 에서 80건을 보냅니다.
2. **바꿔 보기 A, B**: 바꿔 보기 셀에는 바꾼 값이 이미 들어 있습니다. 셀을 그대로 실행해 80건을 다시 보내고, 기본 실행 때와 결과가 어떻게 다른지 봅니다.
3. **Q1, Q2 답 적기**: 질문 아래 답 셀의 「(여기에 적습니다)」를 지우고, 출력에 나온 숫자를 근거로 답을 적습니다.
4. **확장 미션(심화)**: 목적만 받고 접수 요청 분류 · 공유 프로그램을 Claude Code 로 직접 만들어 봅니다.

**미션 재료 80건을 읽고, 모델에게서 담당팀, 위치, 긴급 여부를 받을 스키마와 분류 기준 지시문을 먼저 적습니다.**

- **참고할 예제**: 예제 2-1 (3), (4)
- **✏️ 더 해 보기(선택) 3곳**(코드의 ✏️ 줄 사이): 담당팀 목록 `TEAMS`, 분류 기준 `URGENT_RULE` · `SYS2`, 직접 붙인 정답 라벨 `HAND` 에는 기준이 되는 값이 들어 있습니다. 처음에는 그대로 실행합니다. 기본 미션을 마친 뒤 값을 다르게 바꿔 실습 미션 2 (1) 부터 다시 실행하면 분류 결과와 정답률이 어떻게 달라지는지 볼 수 있습니다. `HAND` 의 라벨은 `TEAMS` 에 있는 이름으로 적습니다.
- 위치 `location` 에는 글자 또는 `null`(값 없음)이 들어갑니다. `null` 을 허용하지 않으면, 본문에 위치가 없어도 모델이 그럴듯한 위치를 지어내서 채웁니다.

In [ ]:
# 실습 미션 2 (1) 스키마를 먼저 적습니다 · 예제 2-1 (3)(4)
m2 = pd.read_csv('2일차_미션2_재료.csv').fillna('')
print(m2['본문'].head(8).to_string())        # 본문 8건을 먼저 읽고 담당팀을 정합니다

# ✏️ ======== 더 해 보기(선택) 1/3 · 담당팀 허용값 · 처음에는 그대로 실행 ========
TEAMS = ['설비', '시설', 'IT', '안전', '요청아님']
# ✏️ ======== 여기까지 ========
SCHEMA2 = {
    'type': 'object',
    'properties': {
        'id':       {'type': 'string'},
        'team':     {'type': 'string', 'enum': TEAMS},
        'location': {'type': ['string', 'null']},     # 위치가 없으면 null
        'urgent':   {'type': 'boolean'},
    },
    'required': ['id', 'team', 'location', 'urgent'],
    'additionalProperties': False,
}
# ✏️ ======== 더 해 보기(선택) 2/3 · 분류 기준 · URGENT_RULE 이 긴급 기준 · 처음에는 그대로 실행 ========
URGENT_RULE = '생산이나 사람 안전에 바로 영향이 있으면 urgent=true.'
SYS2 = ('너는 반도체 공장 접수 요청을 담당팀으로 나누는 도우미다. 장비 고장·알람은 설비, '
        '건물·환경·편의 시설은 시설, PC·시스템은 IT, '
        '사람 안전·보안·약품은 안전, 요청이 아니면 요청아님. '
        'location 은 본문에 나온 장비명이나 장소, 없으면 null. ' + URGENT_RULE)
# ✏️ ======== 여기까지 ========
FMT2 = {'format': {'type': 'json_schema', 'schema': SCHEMA2}}
print(len(m2), '건 · 허용값', TEAMS)

# ✏️ ======== 더 해 보기(선택) 3/3 · 직접 붙인 정답 라벨 · 처음에는 그대로 실행 ========
HAND = {'R001': '설비', 'R007': 'IT', 'R013': '안전', 'R019': '설비', 'R025': '안전',
        'R031': '설비', 'R043': '시설', 'R049': '시설', 'R061': '설비', 'R073': '시설'}
# ✏️ ======== 여기까지 ========
print(m2.set_index('id').loc[list(HAND), '본문'].to_string())   # 라벨 붙일 10건

**한 건을 분류하는 함수를 만들고, 먼저 3건으로 결과를 확인합니다. 결과가 이상하면 실습 미션 2 (1) 의 허용값과 분류 기준을 고친 뒤 (1) 부터 다시 실행합니다.**

- **참고할 예제**: 예제 2-2 (2)
- `run_all()` 은 표에 있는 문의를 8건씩 동시에 보내는 함수입니다. 지시문(`system`)이나 구조화 출력 설정(`fmt`)을 넘기면 넘긴 값으로 보내고, 넘기지 않으면 `SYS2` 와 `FMT2` 로 보냅니다.

In [ ]:
# 실습 미션 2 (2) 3건으로 먼저 결과를 확인합니다 · 예제 2-2 (2)
def classify2(x, system=None, fmt=None):     # 넘기지 않으면 SYS2 · FMT2
    r = ask(system or SYS2, f"id: {x['id']}\n본문: {x['본문']}", output_config=fmt or FMT2)
    o = json.loads(text_of(r))
    o.update(in_tok=r.usage.input_tokens, out_tok=r.usage.output_tokens, stop=r.stop_reason)
    return o

def safe2(x, system=None, fmt=None):
    try:
        return classify2(x, system, fmt)
    except Exception as e:                   # 한 건이 실패해도 배치는 계속 갑니다
        return {'id': x['id'], 'error': type(e).__name__}

def run_all(rows, system=None, fmt=None):    # 8건씩 동시에 보냅니다
    with ThreadPoolExecutor(max_workers=8) as ex:
        return list(ex.map(lambda x: safe2(x, system, fmt),
                           [x for _, x in rows.iterrows()]))

for o in [safe2(x) for _, x in m2.head(3).iterrows()]:
    print(o)
# 3건 결과가 이상하면 (1) 의 TEAMS · SYS2 를 고치고 (1) 부터 다시 실행합니다

**실습 미션 2 (2) 의 3건에 문제가 없으면 80건을 8건씩 동시에 보냅니다. 그다음 보낸 건수와 받은 건수가 같은지, id 가 모두 맞는지 확인합니다.**

- **참고할 예제**: 예제 2-2 (3)
- 80건을 보내는 데 15초 정도 걸립니다. 담당팀별 건수도 함께 출력됩니다.

In [ ]:
# 실습 미션 2 (3) 3건이 괜찮으면 80건을 보냅니다 · 예제 2-2 (3)
rows_m2 = run_all(m2)
r2df = pd.DataFrame(rows_m2)
print('보낸 건수', len(m2), '· 받은 건수', len(r2df), '· id 일치', set(r2df['id']) == set(m2['id']))
print(r2df['team'].value_counts().to_dict())

**80건의 결과를 검증합니다. 검사에 걸린 건에는 그 이유를 적고, 모든 건에 처리 상태를 붙입니다. 처리 상태는 세 가지입니다. `auto` 는 검사를 통과해 그대로 처리하는 건, `review` 는 검사에 걸려 사람이 확인해야 하는 건, `manual` 은 호출이 실패해 사람이 직접 처리해야 하는 건입니다. `review` 와 `manual` 은 엑셀 파일의 「확인필요」 시트에 모읍니다. 결과는 엑셀 파일의 두 시트(전체, 확인필요)와 SQLite DB 에 저장합니다. DB 에는 같은 결과를 두 번 넣고, 행 수가 그대로인지 확인합니다.**

- **참고할 예제**: 예제 2-3 (1), (2)와 예제 2-2 (4), (5), (6)
- **업무 규칙**은 형식은 맞지만 업무상 말이 안 되는 조합을 잡아내는 검사입니다. 예시로 「담당팀이 안전인데 긴급이 아니다」를 넣어 두었습니다.
- **✏️ 더 해 보기(선택)**: 기본 미션을 마친 뒤 같은 모양으로 업무 규칙을 하나 더 넣고 이 셀을 다시 실행해 봅니다(코드의 ✏️ 줄 사이). 예: 담당팀이 요청아님인데 긴급이다.
- `INSERT OR REPLACE` 는 같은 id 의 행이 이미 있으면 새 행을 더하지 않고 그 행을 덮어씁니다. 그래서 두 번 넣어도 행 수는 80 그대로입니다.

In [ ]:
# 실습 미션 2 (4) 검증하고 status 를 붙여 엑셀 두 시트 · DB 에 저장합니다 · 예제 2-3 (1)(2) · 2-2 (4)(5)(6)
def validate2(o, want_id):
    if 'error' in o:                        return '호출 실패'
    if o.get('stop') != 'end_turn':         return '응답 잘림'
    if o.get('id') != want_id:              return 'id 불일치'
    # ✏️ ======== 더 해 보기(선택) · 업무 규칙 · 기본 미션을 마친 뒤 같은 모양으로 하나 더 넣어 봄 ========
    if o['team'] == '안전' and not o['urgent']:   # 업무 규칙 · 예시 하나
        return '안전인데 긴급 아님'
    # 예: if o['team'] == '요청아님' and o['urgent']: return '요청아님인데 긴급'
    # ✏️ ======== 여기까지 ========
    return None

def status2(why):
    # 통과(None)는 표에서 NaN 으로 바뀔 수 있어 글자가 아니면 auto 로 봅니다
    if not isinstance(why, str): return 'auto'
    if why == '호출 실패': return 'manual'
    return 'review'

r2df['why'] = [validate2(o, i) for o, i in zip(rows_m2, m2['id'])]
r2df['status'] = [status2(w) for w in r2df['why']]
print(Counter(r2df['why'].dropna()), '· 통과', r2df['why'].isna().sum(), '/', len(r2df))

m2out = m2.merge(r2df, on='id', how='left')
with pd.ExcelWriter('미션2_결과.xlsx') as w:
    m2out.to_excel(w, sheet_name='전체', index=False)
    m2out[m2out['status'] != 'auto'].to_excel(w, sheet_name='확인필요', index=False)
print({k: len(v) for k, v in pd.read_excel('미션2_결과.xlsx', sheet_name=None).items()})

con2 = sqlite3.connect('mission2.db')
con2.execute('''CREATE TABLE IF NOT EXISTS requests (
    id TEXT PRIMARY KEY, team TEXT, urgent INTEGER, model TEXT, run_at TEXT)''')
db_rows = [(o['id'], o['team'], int(o['urgent']), SMALL_MODEL, RUN_AT) for o in rows_m2 if 'error' not in o]
SQL2 = 'INSERT OR REPLACE INTO requests VALUES (?, ?, ?, ?, ?)'
con2.executemany(SQL2, db_rows)
con2.executemany(SQL2, db_rows)   # 한 번 더 · 행 수가 그대로여야 합니다
con2.commit()
print('두 번 넣은 뒤 행 수', con2.execute('SELECT COUNT(*) FROM requests').fetchone()[0])

**본문 10건에 직접 붙인 담당팀 정답 라벨을 모델 결과와 비교해 정답률을 잽니다. 형식 검증을 통과한 비율과 실제로 맞힌 비율이 다를 수 있다는 점을 확인합니다.**

- **참고할 예제**: 예제 2-3 (2) 에서 건수를 세는 코드
- **✏️ 더 해 보기(선택)**: 기본 미션을 마친 뒤 실습 미션 2 (1) 셀의 `HAND` 를 바꾸고 (1) 셀과 이 (5) 셀만 다시 실행합니다. (2), (3) 셀까지 다시 실행하면 80건을 다시 보내게 되어 모델 결과가 달라질 수 있습니다.
- `hand_hits()` 는 결과 표를 받아서 라벨과 같은 답이 몇 건인지 세는 함수입니다.

In [ ]:
# 실습 미션 2 (5) 직접 붙인 정답 라벨 10건으로 정답률을 잽니다 · 예제 2-3 (2)
def hand_hits(df):                           # 라벨과 같은 답의 수
    got = df.set_index('id')['team']
    return sum(got.get(k) == v for k, v in HAND.items())

print(f"형식 검증 통과율 {r2df['why'].isna().mean() * 100:.1f}% · "
      f"직접 붙인 정답 라벨 정답률 {hand_hits(r2df)}/{len(HAND)}")
got = r2df.set_index('id')
for k, v in HAND.items():
    if got.loc[k, 'team'] != v:
        print(' 갈린 건', k, '라벨', v, '· 모델', got.loc[k, 'team'], '·', m2.set_index('id').loc[k, '본문'][:40])

### 바꿔 보기 A: 허용값에서 담당팀 하나를 빼 봅니다
**허용값에서 'IT' 를 뺀 셀을 그대로 실행하고, IT 로 분류되던 건이 어디로 가는지 봅니다.**

담당팀 허용값에서 'IT' 를 빼고 80건을 다시 보냅니다. 기본 실행에서 IT 로 분류된 건이 어느 팀으로 가는지, 직접 붙인 정답 라벨과 비교한 정답률이 어떻게 바뀌는지 봅니다. 분류 기준 `SYS2` 는 바꾸지 않으므로 IT 를 설명하는 문장은 그대로 남아 있습니다. 그래서 모델은 IT 설명을 읽고도 IT 로는 답할 수 없습니다.
예제 2-1 에서 본 것처럼, 구조화 출력은 허용값(`enum`)에 없는 값을 답으로 돌려주지 않습니다. **✏️ 더 해 보기(선택)**: 먼저 셀을 그대로 실행한 뒤, 셀 맨 위 한 줄에 있는 `DROP_TEAM` 을 다른 팀 이름(예: '안전', '요청아님')으로 바꿔 다시 실행해 볼 수도 있습니다.

In [ ]:
# 실습 미션 2 바꿔 보기 A · 담당팀 허용값에서 하나를 빼고 80건을 다시 보냅니다
# ✏️ ======== 더 해 보기(선택) · 처음에는 그대로 실행, 그다음 값을 바꿔 다시 실행 ========
DROP_TEAM = 'IT'          # 이 값을 바꿔 다시 실행합니다 · 허용값에서 뺄 이름
# ✏️ ======== 여기까지 ========
import copy
SCHEMA_A = copy.deepcopy(SCHEMA2)
SCHEMA_A['properties']['team']['enum'] = [t for t in TEAMS if t != DROP_TEAM]
rA = pd.DataFrame(run_all(m2, fmt={'format': {'type': 'json_schema',
                                           'schema': SCHEMA_A}}))
print(rA['team'].value_counts().to_dict())
base = r2df.set_index('id')['team']
was = base[base == DROP_TEAM].index          # 기본 실행에서 그 팀이던 건
moved = rA.set_index('id').loc[was, 'team']
print(f'기본 실행에서 {DROP_TEAM} 였던 건', len(was), '→', moved.value_counts().to_dict())
for i, t in moved.items():
    print(' ', i, DROP_TEAM, '→', t, '·', m2.set_index('id').loc[i, '본문'][:40])
print('라벨 정답률', hand_hits(rA), '/', len(HAND), f'(기본 실행 {hand_hits(r2df)})')

**Q1.** 담당팀 허용값에서 'IT' 를 뺐을 때의 결과를 보고 답해 주세요.
1. 기본 실행에서 IT 로 분류된 건은 어느 팀으로 몇 건씩 갔나요?
2. 직접 붙인 정답 라벨 10건과 비교한 정답률은 얼마에서 얼마로 바뀌었나요?
3. 실제 업무에서는 허용값 목록에 맞는 팀이 없는 요청도 들어올 수 있습니다. 이런 경우에 대비해 허용값 목록을 어떻게 정하겠는지 적어 주세요.

**답** · ✏️ 답 적는 곳 (두 번 클릭해 적기)

(여기에 적습니다)

### 바꿔 보기 B: 분류 기준에서 긴급 기준 문장을 빼 봅니다
**긴급 기준 문장을 뺀 셀을 그대로 실행하고, 긴급 건수와 확인필요 건수가 어떻게 바뀌는지 봅니다.**

분류 기준 `SYS2` 에서 긴급 기준 문장 `URGENT_RULE` 을 빼고 80건을 다시 보냅니다. 모델이 긴급(`urgent` 가 true)으로 답한 건수와, 실습 미션 2 (4) 의 검증에서 「확인필요」로 분류되는 건수가 어떻게 바뀌는지 봅니다. 긴급 판단이 바뀐 건은 본문 앞부분과 함께 출력됩니다.
**✏️ 더 해 보기(선택)**: 먼저 셀을 그대로 실행한 뒤, 셀 맨 위 한 줄의 `DROP_RULE` 에 `SYS2` 의 다른 문장을 넣어 다시 실행해 볼 수도 있습니다. 이때 넣는 문장은 `SYS2` 에 있는 문장과 글자 하나까지 같아야 빠집니다. 출력 첫 줄이 `False` 이면 문장이 빠지지 않은 것이므로, `SYS2` 에서 문장을 그대로 복사해 따옴표 안에 넣습니다.

In [ ]:
# 실습 미션 2 바꿔 보기 B · 분류 기준에서 한 문장을 빼고 80건을 다시 보냅니다
# ✏️ ======== 더 해 보기(선택) · 처음에는 그대로 실행, 그다음 값을 바꿔 다시 실행 ========
DROP_RULE = URGENT_RULE   # 이 값을 바꿔 다시 실행합니다 · SYS2 에서 뺄 문장
# ✏️ ======== 여기까지 ========
print('SYS2 에서 뺀 문장 찾음:', DROP_RULE in SYS2)   # False 면 아무것도 빠지지 않습니다
rows_b = run_all(m2, system=SYS2.replace(DROP_RULE, ''))
rB = pd.DataFrame(rows_b)
why_b = [validate2(o, i) for o, i in zip(rows_b, m2['id'])]
need_base = (r2df['status'] != 'auto').sum()
need_b = sum(status2(w) != 'auto' for w in why_b)
print('urgent true', (rB['urgent'] == True).sum(), f"(기본 실행 {(r2df['urgent'] == True).sum()})")
print('확인필요', need_b, f'(기본 실행 {need_base})', Counter(w for w in why_b if w))
u0, u1 = r2df.set_index('id')['urgent'], rB.set_index('id')['urgent']
ch = [i for i in m2['id'] if u0.get(i) != u1.get(i)]    # 긴급 판단이 바뀐 건
print('긴급 판단이 바뀐 건', len(ch), '· 앞 8건')
for i in ch[:8]:
    print(' ', i, '기본', u0.get(i), '→', u1.get(i), '·', m2.set_index('id').loc[i, '본문'][:40])

**Q2.** 분류 기준에서 긴급 기준 문장(`URGENT_RULE`)을 뺐을 때의 결과를 보고 답해 주세요.
1. `urgent` 가 true 인 건수와 「확인필요」 건수는 각각 얼마에서 얼마로 바뀌었나요?
2. 늘어난 확인필요 건은 실습 미션 2 (4) 의 어느 업무 규칙에 걸렸나요?
3. 이 분류 결과로 실제 긴급 알림을 보낸다면, 긴급 기준 문장을 어떻게 고쳐 쓰겠는지 적어 주세요.

**답** · ✏️ 답 적는 곳 (두 번 클릭해 적기)

(여기에 적습니다)

### 확장 미션 2 (심화) · 접수 요청 분류 · 공유 프로그램 만들기
**기본 미션을 마친 경우에만 합니다. 목적만 보고 어떻게 만들지 스스로 정한 뒤, Claude Code 로 직접 만들어 봅니다.**

**목적**
설비, 시설, IT, 안전 요청이 한 창구로 섞여 들어옵니다. 요청을 담당팀별로 나눠 엑셀로 공유하고 DB 에도 쌓아, 사람이 확인해야 할 건만 따로 볼 수 있게 하고 싶습니다.

**재료**: `2일차_미션2_재료.csv` (코랩 왼쪽 파일 패널에서 ⋮ → [다운로드]) · 본인 업무에 맞게 재료를 새로 만들어 진행해도 됩니다.

**만들기 전에 생각해 볼 것**
- 모델에게서 어떤 항목을 어떤 모양으로 받아야 할까요?
- 사람이 확인해야 할 건은 어떤 기준으로 골라낼까요?
- 같은 접수 파일로 프로그램을 두 번 돌려도 DB 에 같은 요청이 두 번 쌓이지 않게 하려면 어떻게 해야 할까요?

**진행**
1. 목적을 읽고, 어떤 단계로 만들지 먼저 짧게 메모합니다.
2. PC 에 작업 폴더를 만들고 재료 파일을 넣은 뒤, Claude Code 에서 그 폴더를 열어 **새 세션**을 시작합니다.
3. 목적과 내 계획을 Claude Code 에 보내 만들게 하고, 결과를 보며 고쳐 달라고 요청하면서 완성해 갑니다.
4. 다 만들면 맨 아래 「회고」에 직접 만들어 보니 어땠는지 적습니다.

**✏️ 회고** · 직접 만들어 보니 어땠는지 자유롭게 적습니다. 예를 들면 이런 것들입니다.
- 처음 세운 계획과 실제로 만들어진 구조가 어떻게 달랐는지
- Claude Code 에 다시 요청한 것과 그 까닭
- 오늘 미션의 코드와 다르게 만든 부분, 또는 미션 코드에서 가져다 쓴 부분

**답** · ✏️ 답 적는 곳 (두 번 클릭해 적기)

(여기에 적습니다)

## 미션 3 준비: 문서 253건을 검색할 준비를 다시 합니다
**검색 준비를 다시 만드는 셀입니다. 실행만 합니다.**

예제 3-2 (1)과 예제 3-3 (1), (3), (7) 의 코드를 이 셀 하나에 모았습니다. 임베딩 모델을 처음 내려받을 때는 1~2분 걸립니다.
- `docs` 와 `names` 는 예제 3-2 (1), `model` 은 예제 3-3 (1), `E` 는 예제 3-3 (3), `search()` 는 예제 3-3 (7) 에서 만들었습니다.

`검색 준비 완료` 가 출력되면 미션 3 을 시작합니다.

In [ ]:
# 미션 3 준비 · 예제 3-2 · 3-3 의 검색 준비
import numpy as np
from sentence_transformers import SentenceTransformer
files = [f for f in sorted(os.listdir('corpus')) if f.endswith('.md')]   # 예제 3-2 (1)
docs  = [open(f'corpus/{f}', encoding='utf-8').read() for f in files]
names = [f[:-3] for f in files]
model = SentenceTransformer('intfloat/multilingual-e5-small')          # 예제 3-3 (1) · 임베딩 모델
E = model.encode(['passage: ' + d for d in docs], normalize_embeddings=True, batch_size=32)   # 예제 3-3 (3)
def search(q, k=3):                                  # 예제 3-3 (7)
    v = model.encode(['query: ' + q], normalize_embeddings=True)[0]
    sim = E @ v
    return [(names[i], round(float(sim[i]), 3)) for i in sim.argsort()[::-1][:k]]
print('검색 준비 완료 ·', len(docs), '건 ·', E.shape)

## 실습 미션 3 · 질문 3개로 임베딩 검색 1위 판정하기
**질문 3개를 검색하고, 1위 문서가 질문에 답하는지 직접 판정합니다. 판정이 어려우면 판정 도우미로 모델에게 물어봅니다.**

재료 `2일차_미션3_재료.txt` 에 있는 현장 질문 가운데 3개(재료의 1, 4, 5번)를 `search()` 로 검색합니다. 그리고 1위로 나온 문서를 직접 열어 읽고, 그 문서가 질문에 답하는 문서인지 판정합니다.
예제 3-2 에서 만든 `docs` 와 `names`, 예제 3-3 에서 만든 `model`, `E`, `search()` 를 그대로 씁니다. 이것들은 바로 위의 「미션 3 준비」 셀이 다시 만들어 둡니다.

**✏️ 이 미션에서 손댈 곳**
- 직접 고칠 곳: 실습 미션 3 (2) 셀의 `NAME`(열어 볼 문서 이름), (3) 셀의 `JUDGE`(판정) · `RANK`(정답 문서 순위)
- 판정이 어려울 때(선택): (2-1) 판정 도우미의 `QUESTION_NUMBER` · `DOC_NAME`
- 답 적는 곳: Q1, Q2
- 더 해 보기(선택): 바꿔 보기 A 의 `Q_A`, 바꿔 보기 B 의 `Q_B`

**진행 순서**
1. **기본 실행**: (1)~(3) 셀에서 질문 3개를 검색하고, 1위 문서를 열어 읽은 뒤, 판정 결과를 적습니다. 기본 실행에서는 질문 문장을 고치지 않습니다.
2. **바꿔 보기 A, B**: 바꿔 보기 셀에는 고쳐 쓴 질문 문장이 이미 들어 있습니다. 셀을 그대로 실행해 다시 검색하고, 1위 문서와 점수가 어떻게 바뀌는지 봅니다.
3. **Q1, Q2 답 적기**: 질문 아래 답 셀의 「(여기에 적습니다)」를 지우고, 출력에 나온 문서 이름과 점수를 근거로 답을 적습니다.
4. **확장 미션(심화)**: 목적만 받고 사내 문서 질의응답 도구를 Claude Code 로 직접 만들어 봅니다.

**질문 3개를 검색해, 질문마다 1~3위 문서의 이름, 점수, 제목을 봅니다.**

- **참고할 예제**: 예제 3-3 (7) 의 `search()`
- 질문은 재료 파일의 1, 4, 5번 문장을 그대로 넣어 두었습니다. 이 노트북에서는 1, 2, 3번으로 부릅니다.

In [ ]:
# 실습 미션 3 (1) 질문 3개를 검색해 1~3위 문서를 봅니다 · 예제 3-3 (7) 의 search()
MY_Q = [                                    # 재료 파일의 1 · 4 · 5번 질문
    '칠러 유량이 얼마 이상이어야 정상인가요?',
    ('PHOTO-02 파티클이 올라서 필터를 바꾸기로 했다는데, '
     '포토 시즈닝 런은 원래 어떻게 하나요?'),
    ('CMP-02 RCP-CMP-330 이탈 로트가 있었다는데, '
     'CMP 알람 뜨면 1차로 뭘 봐야 하나요?'),
]
for i, q in enumerate(MY_Q, 1):            # 질문 번호는 1, 2, 3
    print(i, q)
    for rank, (name, score) in enumerate(search(q, k=3), 1):
        title = docs[names.index(name)].splitlines()[0].lstrip('# ')   # 첫 줄 = 제목
        print(f'   {rank}위 {score}  {name} · {title[:45]}')


**문서 이름을 넣어 문서 전체를 열고, 질문에 답이 되는 내용(수치나 절차)이 있는지 읽습니다. 제목만 보고 판정하지 않습니다.**

- **✏️ 직접 고칠 곳**: `NAME` 에 실습 미션 3 (1) 출력의 문서 이름을 넣고, 바꿔 가며 여러 번 실행합니다.
- **판정이 어려우면**: 아래 (2-1) 판정 도우미에 질문 번호와 문서 이름을 넣고 실행합니다. 모델이 「답할 수 있음 / 없음」과 근거 문장을 돌려줍니다.

In [ ]:
# 실습 미션 3 (2) 문서 이름을 넣어 문서 전체를 엽니다 · 이름을 바꿔 가며 여러 번 실행합니다
# ✏️ ======== 직접 고칠 곳 · (1) 출력의 문서 이름을 넣음 ========
NAME = 'QIR-2026-0712'                     # 열어 볼 문서 이름
# ✏️ ======== 여기까지 ========
print(docs[names.index(NAME)])             # 문서 전체 · 왼쪽 파일 패널의 corpus 폴더에서 같은 이름의 파일을 열어도 됩니다


In [ ]:
# 실습 미션 3 (2-1) 판정 도우미(선택) · 질문과 문서 전체를 모델에게 주고, 이 문서로 답할 수 있는지 묻습니다
# ✏️ ======== 직접 고칠 곳 · 판정할 질문 번호(1~3)와 문서 이름 ========
QUESTION_NUMBER = 1
DOC_NAME = 'QIR-2026-0712'
# ✏️ ======== 여기까지 ========
document_text = docs[names.index(DOC_NAME)]
judge_request = (f'질문: {MY_Q[QUESTION_NUMBER - 1]}\n\n[문서 {DOC_NAME}]\n{document_text}\n\n'
                 '이 문서만으로 질문에 답할 수 있으면 첫 줄에 「답할 수 있음」, 없으면 「답할 수 없음」이라고 쓴다. '
                 '둘째 줄부터는 판단의 근거가 된 문장을 문서에서 그대로 옮겨 적는다.')
judge_response = client.messages.create(model=MODEL, max_tokens=400,
                                        messages=[{'role': 'user', 'content': judge_request}])
print(text_of(judge_response))


**질문마다 1위 문서가 그 질문에 답하면 `'O'`, 답하지 못하면 `'X'` 를 적고 표로 모읍니다. `'X'` 인 질문은 2위와 3위 문서도 실습 미션 3 (2) 로 열어 보고, 정답 문서가 몇 위에 있는지 적습니다.**

- **✏️ 직접 고칠 곳**: `JUDGE` 에 질문 1~3 의 판정을 모두 적습니다. 아직 적지 않은 질문은 `?` 로 출력됩니다.
- `RANK` 에는 `'X'` 로 판정한 질문만 `{질문 번호: 정답 문서의 순위}` 로 적습니다. 정답 문서가 3위 안에 없으면 `None` 을 적습니다.

In [ ]:
# 실습 미션 3 (3) 판정을 적고 표로 모읍니다 · 1위 문서를 읽고 각자 적습니다
# ✏️ ======== 직접 고칠 곳 · 질문 1~3 의 판정과, X 인 질문의 정답 문서 순위 ========
JUDGE = {}                      # 적는 모양 예: JUDGE = {1: 'O', 2: 'X', 3: 'O'} · 1위 문서가 답하면 'O', 아니면 'X'
RANK = {}                       # 적는 모양 예: RANK = {2: 3} · 'X' 인 질문만 · 정답 문서가 3위면 3, 3위 안에 없으면 None
# ✏️ ======== 여기까지 ========
for i, q in enumerate(MY_Q, 1):
    top, score = search(q, k=1)[0]
    j = JUDGE.get(i, '?')
    more = f' · 정답 문서 순위 {RANK.get(i)}' if j == 'X' else ''
    print(j, score, top, '·', q[:32] + more)
print('1위가 맞은 질문', sum(v == 'O' for v in JUDGE.values()), '/', len(MY_Q))

### 바꿔 보기 A: 질문 앞에 붙은 사례 설명을 지워 봅니다
**2, 3번 질문에서 사례 설명을 지운 셀을 그대로 실행하고, 1위 문서가 바뀌는지 봅니다.**

2번과 3번 질문은 앞부분에 사례 설명(PHOTO-02 필터 교체, CMP-02 이탈 로트)이 붙어 있습니다. 실제로 묻는 뒷부분만 남겨 다시 검색하고, 1위 문서가 바뀌는지 봅니다.
**✏️ 더 해 보기(선택)**: 먼저 셀을 그대로 실행한 뒤, 바꿔 보기부터는 질문 문장을 고쳐 써도 됩니다. `Q_A` 의 문장을 바꾸거나, 다른 번호의 질문을 더해 다시 실행해 볼 수 있습니다.

In [ ]:
# 실습 미션 3 바꿔 보기 A · 사례 이야기를 지운 질문으로 다시 검색합니다
# ✏️ ======== 더 해 보기(선택) · 처음에는 그대로 실행, 그다음 값을 바꿔 다시 실행 ========
Q_A = {2: '포토 시즈닝 런은 원래 어떻게 하나요?',      # 이 문장을 바꿔 다시 실행합니다
       3: 'CMP 알람 뜨면 1차로 뭘 봐야 하나요?'}       # 키는 (1) 의 질문 번호(1~3)
# ✏️ ======== 여기까지 ========
for i, q in Q_A.items():
    print(i, '원래 질문 1위', search(MY_Q[i - 1], k=1)[0])
    print('  바꾼 질문', q)
    for rank, (name, score) in enumerate(search(q, k=3), 1):
        print(f'   {rank}위 {score}  {name}')

**Q1.** 2번(포토 시즈닝 런) 질문과 3번(CMP 알람 1차 대응) 질문에서 앞의 사례 설명을 지웠을 때의 결과를 보고 답해 주세요.
1. 두 질문의 1위 문서는 각각 무엇에서 무엇으로 바뀌었나요?
2. 원래 1위였던 문서에는 질문 앞부분(사례 설명)과 겹치는 단어가 있습니다. 어떤 단어가 겹쳤나요?
3. 현장 담당자가 이 검색 결과를 답의 근거로 쓴다면, 1위 문서 말고 무엇을 더 확인해야 하는지 적어 주세요.

**답** · ✏️ 답 적는 곳 (두 번 클릭해 적기)

(여기에 적습니다)

### 바꿔 보기 B: 질문을 문서 제목처럼 바꿔 써 봅니다
**1, 3번 질문을 문서 제목처럼 바꾼 셀을 그대로 실행하고, 정답 문서의 순위가 어떻게 바뀌는지 봅니다.**

현장에서 말하듯 쓴 질문을, 문서 제목에 쓰일 만한 말(예: 「CMP 장비 알람 1차 대응」)로 바꿔 다시 검색합니다. 1위 점수와 정답 문서의 순위가 원래 질문 때와 어떻게 다른지 봅니다.
**✏️ 더 해 보기(선택)**: 먼저 셀을 그대로 실행한 뒤, `Q_B` 의 문장을 바꾸거나, 다른 번호의 질문을 더해 다시 실행해 볼 수 있습니다.

In [ ]:
# 실습 미션 3 바꿔 보기 B · 질문을 문서 말투로 바꿔 다시 검색합니다
# ✏️ ======== 더 해 보기(선택) · 처음에는 그대로 실행, 그다음 값을 바꿔 다시 실행 ========
Q_B = {1: '에치 장비 칠러 유량 기준',                # 이 문장을 바꿔 다시 실행합니다
       3: 'CMP 장비 알람 1차 대응'}                  # 키는 (1) 의 질문 번호(1~3)
# ✏️ ======== 여기까지 ========
for i, q in Q_B.items():
    print(i, '원래', MY_Q[i - 1][-30:], '→', search(MY_Q[i - 1], k=3))
    print('  바꿈', q, '→', search(q, k=3))

**Q2.** 1번(칠러 유량) 질문과 3번(CMP 알람 1차 대응) 질문을 문서 제목처럼 바꿔 썼을 때의 결과를 보고 답해 주세요.
1. 정답 문서(1번은 SOP-ETCH-007_v2.1, 3번은 SOP-CMP-021)의 순위는 각각 몇 위에서 몇 위로 바뀌었나요?
2. 바꾼 두 질문에서 1위와 2위의 점수 차는 각각 얼마였나요?

**답** · ✏️ 답 적는 곳 (두 번 클릭해 적기)

(여기에 적습니다)

### 확장 미션 3 (심화) · 사내 문서 질의응답 도구 만들기
**기본 미션을 마친 경우에만 합니다. 목적만 보고 어떻게 만들지 스스로 정한 뒤, Claude Code 로 직접 만들어 봅니다.**

**목적**
현장 질문을 넣으면 사내 문서를 근거로 답해 주는 도구가 필요합니다. 답과 함께 어느 문서에서 나온 답인지 보여 줘야 합니다.

**재료**: `corpus` 폴더(문서 253건)와 `corpus_manifest.csv` (코랩 왼쪽 파일 패널에서 다운로드) · 본인 업무에 맞게 재료를 새로 만들어 진행해도 됩니다.

**만들기 전에 생각해 볼 것**
- 질문마다 문서 253건을 매번 처리하지 않으려면 무엇을 미리 만들어 둘까요?
- 1위 문서가 질문에 맞지 않을 때를 어떻게 줄이고, 어떻게 알아챌까요?
- 문서에 답이 없을 때 도구가 어떻게 답해야 할까요?

**진행**
1. 목적을 읽고, 어떤 단계로 만들지 먼저 짧게 메모합니다.
2. PC 에 작업 폴더를 만들고 재료 파일을 넣은 뒤, Claude Code 에서 그 폴더를 열어 **새 세션**을 시작합니다.
3. 목적과 내 계획을 Claude Code 에 보내 만들게 하고, 결과를 보며 고쳐 달라고 요청하면서 완성해 갑니다.
4. 다 만들면 맨 아래 「회고」에 직접 만들어 보니 어땠는지 적습니다.

**✏️ 회고** · 직접 만들어 보니 어땠는지 자유롭게 적습니다. 예를 들면 이런 것들입니다.
- 처음 세운 계획과 실제로 만들어진 구조가 어떻게 달랐는지
- Claude Code 에 다시 요청한 것과 그 까닭
- 오늘 미션의 코드와 다르게 만든 부분, 또는 미션 코드에서 가져다 쓴 부분

**답** · ✏️ 답 적는 곳 (두 번 클릭해 적기)

(여기에 적습니다)

## 마치며
**미션 1, 2, 3 을 마친 노트북 하나를 zip 으로 묶어 휴넷 랩스에 올립니다.**

**제출하는 법**
1. Q1, Q2 답 셀을 모두 적었는지 확인합니다. 확장 미션을 했다면 「회고」 아래 답 셀도 확인합니다.
2. [파일] → [다운로드] → [.ipynb 다운로드] 를 눌러, 실행 결과가 남은 노트북을 내려받습니다.
3. 파일 이름을 `아이디_2일차_실습미션.ipynb` 로 바꿉니다. 예: `aaa_2일차_실습미션.ipynb`
4. zip 으로 묶어 휴넷 랩스(https://dssw.labs.hunet.com/)에 올립니다. 휴넷 랩스에는 .ipynb 파일이 바로 올라가지 않습니다.